# <font color="steelblue">23. Variables aleatorias discretas: binomial y Poisson</font>

*Unidad 4. Probabilidad* · Sesión teórica 23 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (ninguno) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = []
PREGUNTAS = ["q_P23_1.json", "q_P23_2.json", "q_P23_3.json", "q_P23_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Si el 30 % de la población lee prensa digital y entrevistas a 20 personas, ¿cuántas lo harán? Si una agencia recibe de media 4 alertas de bulos por hora, ¿qué probabilidad hay de recibir 10 en una hora? Estas preguntas se responden con **modelos de probabilidad**: la **binomial** para contar éxitos en un número fijo de intentos y la **Poisson** para contar sucesos en un intervalo de tiempo.

**Qué necesitas saber antes**: los cuadernos 20 y 21.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Explicar qué es una variable aleatoria discreta y su distribución.
2. Calcular probabilidades con la distribución binomial.
3. Calcular probabilidades con la distribución de Poisson.
4. Usar `scipy.stats` y comprobarlo con simulación.

## <font color='steelblue'>Contenidos</font>

1. Variables aleatorias
2. La distribución binomial
3. Comprobación por simulación
4. La distribución de Poisson

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Variables aleatorias</font>

Una **variable aleatoria** asigna un número a cada resultado de un experimento aleatorio: el número de personas que leen prensa en una muestra, el número de alertas en una hora. Es **discreta** si toma valores que se cuentan (0, 1, 2...).

Su **distribución** dice la probabilidad de cada valor, y tiene una **media** (o esperanza, lo que sale «en promedio») y una **desviación típica**.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y marca **todas** las respuestas correctas:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P23_1.json")

## <font color='steelblue'>2. La distribución binomial</font>

La **binomial** $B(n, p)$ cuenta el número de **éxitos** en $n$ intentos **independientes**, cada uno con probabilidad $p$ de éxito.

- Media: $n \cdot p$. Desviación típica: $\sqrt{n\,p\,(1-p)}$.
- Ejemplo: número de personas que leen prensa digital entre 20 entrevistadas, si el 30 % lo hace: $B(20; 0{,}3)$.

In [ ]:
n, p = 20, 0.3
X = stats.binom(n, p)
print("Media:", X.mean(), "  Desviación típica:", round(X.std(), 2))
print("P(X = 6)  =", round(X.pmf(6), 4))        # pmf: probabilidad de un valor concreto
print("P(X <= 3) =", round(X.cdf(3), 4))        # cdf: probabilidad acumulada
print("P(X >= 10) =", round(1 - X.cdf(9), 4))

In [ ]:
valores = np.arange(0, n + 1)
plt.bar(valores, X.pmf(valores), color="steelblue")
plt.xlabel("Número de lectores de prensa digital entre 20 personas")
plt.ylabel("Probabilidad")
plt.title("Distribución binomial B(20; 0,3)")
plt.show()

**Cómo leerlo**: lo más probable es encontrar unas 6 personas (la media), pero obtener 3 o 9 tampoco es raro. Por eso **una muestra pequeña puede dar porcentajes muy distintos del real**.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P23_2.json")

## <font color='steelblue'>3. Comprobación por simulación</font>

Simulamos 10.000 encuestas de 20 personas y comparamos con la fórmula:

In [ ]:
rng = np.random.default_rng(23)
muestras = rng.binomial(n, p, size=10_000)
print("P(X <= 3) simulada:", (muestras <= 3).mean(), "   exacta:", round(X.cdf(3), 4))

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P23_3.json")

## <font color='steelblue'>4. La distribución de Poisson</font>

La **Poisson** $P(\lambda)$ cuenta sucesos que ocurren **al azar en un intervalo** (de tiempo o de espacio), con una media de $\lambda$ por intervalo: alertas por hora, accidentes por fin de semana, cartas al director por día.

- Media y varianza: $\lambda$.

In [ ]:
lam = 4
Y = stats.poisson(lam)
print("P(Y = 4)   =", round(Y.pmf(4), 4))
print("P(Y >= 10) =", round(1 - Y.cdf(9), 5))
vals = np.arange(0, 15)
plt.bar(vals, Y.pmf(vals), color="darkorange")
plt.xlabel("Alertas de bulos en una hora")
plt.ylabel("Probabilidad")
plt.title("Poisson con media 4 alertas por hora")
plt.show()

**Para qué sirve en una redacción**: si un fin de semana hay 9 accidentes mortales en una provincia donde la media es 4, ¿es algo excepcional? Con la Poisson, P(Y ≥ 9) ≈ 2 %: es raro, pero sucede alguna vez al año por puro azar. Antes de hablar de «repunte» hay que mirar más fines de semana.

In [ ]:
print("P(9 o más accidentes | media 4) =", round(1 - stats.poisson(4).cdf(8), 4))

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P23_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 2)*

En un municipio, el 60 % de los hogares tiene una suscripción digital. Si eliges 15 hogares al azar, ¿qué probabilidad hay de que **menos de la mitad** (7 o menos) tengan suscripción?

<details><summary>Ver solución</summary>

```python
stats.binom(15, 0.6).cdf(7)
```
</details>

### **Ejercicio 2** *(sección 2)*

Dibuja las distribuciones B(10; 0,5) y B(100; 0,5) en dos gráficos. ¿Qué cambia en la forma?

<details><summary>Ver solución</summary>

```python
for n_ in [10, 100]:
    v = np.arange(0, n_ + 1)
    plt.bar(v, stats.binom(n_, 0.5).pmf(v))
    plt.title(f"B({n_}; 0,5)")
    plt.show()
```

Con n grande la forma se parece a una campana: es la distribución normal del siguiente cuaderno.
</details>

### **Ejercicio 3** *(sección 4)*

Una web recibe de media 2,5 denuncias de usuarios por día. ¿Qué probabilidad hay de que un día no reciba ninguna? ¿Y de que reciba 6 o más?

<details><summary>Ver solución</summary>

```python
Z = stats.poisson(2.5)
print(Z.pmf(0), 1 - Z.cdf(5))
```
</details>

### **Ejercicio 4** *(sección 4)*

Simula 52 fines de semana con una media de 4 accidentes. ¿Cuántos tienen 9 o más?

<details><summary>Ver solución</summary>

```python
(rng.poisson(4, size=52) >= 9).sum()
```
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- Una **variable aleatoria discreta** toma valores que se cuentan; su **distribución** da la probabilidad de cada uno.
- **Binomial** $B(n, p)$: éxitos en $n$ intentos independientes; media $np$.
- **Poisson** $P(\lambda)$: sucesos al azar en un intervalo; media $\lambda$.
- En `scipy.stats`: `pmf` (probabilidad de un valor), `cdf` (acumulada) y `1 - cdf` (cola superior).
- Un valor «excepcional» puede ser puro azar: calcula su probabilidad antes de titular «repunte».

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Variable aleatoria** | Número asociado al resultado de un experimento aleatorio. |
| **Binomial** | Distribución del número de éxitos en n intentos independientes. |
| **Poisson** | Distribución del número de sucesos en un intervalo. |
| **Esperanza** | Media de una variable aleatoria. |
| **`pmf` / `cdf`** | Función de probabilidad / función de distribución acumulada. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `P(X >= k)` sale mal | Se ha usado `1 - cdf(k)` | En discretas, P(X ≥ k) = 1 − cdf(k − 1) |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 24 estudiarás la distribución más importante: la normal.

**Siguiente cuaderno**: [24. La distribución normal](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P24_distribucion_normal.ipynb)